# Fine-tuning Gemma 4 E4B on the CBU Faculty Manual
### Apple Silicon · MLX · LoRA · Ollama

This notebook walks through the full pipeline for fine-tuning a local language model on the
California Baptist University Employee Manual (Faculty Section) — entirely on Apple Silicon,
no cloud GPU required.

**Pipeline overview**

```
PDF  →  Chunks  →  Enrich  →  QA Pairs  →  Format  →  Train  →  Evaluate  →  Export GGUF
 1          2          3           4            5        6           7
```

| Stage | Script | Output |
|-------|--------|--------|
| 1 — Chunk | `chunker.py` | `data/chunks.jsonl` — 90 policy sections |
| 2 — Enrich | `enricher.py` | `data/enriched.jsonl` — metadata-tagged chunks |
| 3 — Generate | `generator.py` | `data/seeds.jsonl` — 300 QA pairs via Ollama |
| 4 — Format | `formatter.py` | `data/train.jsonl` + `data/valid.jsonl` |
| 5 — Train | `trainer.py` | `outputs/cbu-gemma4-e4b-mlx-v1/` — LoRA adapter |
| 6 — Evaluate | `evaluator.py` | `eval/results.jsonl` — ROUGE-L scores |
| 7 — Plot | `plot_eval.py` | `eval/rouge_chart.png` — per-type bar chart |
| 8 — Export | `burn_gguf.py` | `burns/` — Q4_K_M GGUF for Ollama |

**Prerequisites**
- Mac with Apple Silicon (M1 / M2 / M3 / M4)
- [Ollama](https://ollama.com) installed and running with `gemma4:latest` pulled
- `cbu_faculty.pdf` in the project root
- [Homebrew llama.cpp](https://formulae.brew.sh/formula/llama.cpp) for GGUF export (Step 8 only)
- Conda environment or virtualenv with packages installed (Step 0)

---
## Step 0 — Install Packages

Run once, then proceed. If using the `caimll_finetuning` conda environment, these are already installed.

In [ ]:
!pip install mlx-lm requests pypdf rouge-score matplotlib numpy

---
## Step 1 — Verify Environment

Confirm MLX can see the GPU and Ollama is reachable before running any stage.

In [ ]:
import platform, subprocess, sys
import mlx.core as mx
import requests

print(f"Python       : {sys.version.split()[0]}")
print(f"Platform     : {platform.platform()}")

# MLX device
device = mx.default_device()
print(f"MLX device   : {device}")
assert str(device) == 'Device(gpu, 0)', "MLX is not using the GPU — check Apple Silicon environment"

# MLX version
import mlx
print(f"MLX version  : {mlx.__version__}")

# Ollama
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    gemma4_available = any('gemma4' in m for m in models)
    print(f"Ollama       : running  (gemma4: {'✓' if gemma4_available else '✗ — run: ollama pull gemma4:latest'})")
except Exception:
    print("Ollama       : NOT reachable — start with: ollama serve")

# PDF
from pathlib import Path
pdf = Path('cbu_faculty.pdf')
print(f"PDF          : {'✓ ' + str(round(pdf.stat().st_size/1e6,1)) + ' MB' if pdf.exists() else '✗ cbu_faculty.pdf not found'}")

print("\nEnvironment check complete.")

---
## Step 2 — Extract PDF → Policy Chunks

**`chunker.py`** parses the CBU Faculty Manual PDF and splits it into policy-level sections.
Each chunk maps to one policy number (`3.xxx` or `4.x`) and includes metadata:
policy title, category, effective date, word count, and cross-references.

Detection strategy: scans pages line-by-line for standalone policy numbers
on pages that contain `Effective Date:` or `Updated:` headers — robust to CBU's varied PDF formatting.

Expected output: **~90 chunks across 11 categories**

In [ ]:
%run scripts/chunker.py

In [ ]:
import json
from collections import Counter

chunks = [json.loads(l) for l in open('data/chunks.jsonl')]
cats = Counter(c['category'] for c in chunks)

print(f"Total chunks : {len(chunks)}")
print(f"Avg words    : {sum(c['word_count'] for c in chunks) // len(chunks)}")
print()
print(f"{'Category':<35} {'Chunks':>6}")
print("-" * 43)
for cat, n in sorted(cats.items(), key=lambda x: -x[1]):
    print(f"{cat:<35} {n:>6}")

print("\n── Sample chunk ──")
s = chunks[0]
print(f"Policy   : {s['policy_num']} — {s['policy_title']}")
print(f"Category : {s['category']}")
print(f"Words    : {s['word_count']}")
print(f"Text     : {s['text'][:300]}...")

---
## Step 3 — Enrich Chunks with Metadata

**`enricher.py`** tags each chunk with additional metadata used to guide QA generation:
audience type (tenure-track, adjunct, all), topic tags, and key entities.
Uses a keyword heuristic with Ollama as an optional fallback for ambiguous chunks.

Expected output: **90 enriched chunks** (same count, added fields)

In [ ]:
%run scripts/enricher.py

In [ ]:
enriched = [json.loads(l) for l in open('data/enriched.jsonl')]

audience_counts = Counter(c.get('audience', 'unknown') for c in enriched)
print(f"Enriched chunks : {len(enriched)}")
print()
print("Audience distribution:")
for aud, n in sorted(audience_counts.items(), key=lambda x: -x[1]):
    print(f"  {aud:<20} {n}")

print("\n── Sample enriched chunk (added fields) ──")
s = enriched[0]
for k in ['audience', 'topic_tags', 'key_entities']:
    if k in s:
        print(f"  {k:<16}: {s[k]}")

---
## Step 4 — Generate QA Pairs via Ollama

**`generator.py`** calls Ollama (`gemma4:latest`) to generate question-answer pairs from each chunk.
Each QA pair includes:
- A question with one of 8 types: `factual`, `procedural`, `eligibility`, `scenario`,
  `rights_responsibilities`, `comparative`, `timeline_deadline`, `committee_role`
- A reference answer ending with `[Reference: CBU Faculty Manual, Policy X.XXX]`
- A reasoning trace (`thought`) explaining how the answer was derived

⚠️ **This step calls Ollama ~300+ times and takes 30–60 minutes depending on hardware.**
The generator is resume-safe — it appends to `seeds.jsonl`, so you can interrupt and restart.

Expected output: **300 QA pairs**

In [ ]:
%run scripts/generator.py --target 300 --model gemma4:latest

In [ ]:
seeds = [json.loads(l) for l in open('data/seeds.jsonl')]
type_counts = Counter(s['question_type'] for s in seeds)

print(f"Total QA pairs : {len(seeds)}")
print()
print(f"{'Question Type':<28} {'Count':>5}")
print("-" * 35)
for qt, n in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {qt:<26} {n:>5}")

print("\n── Sample QA pair ──")
s = seeds[0]
print(f"Type     : {s['question_type']}")
print(f"Question : {s['question']}")
print(f"Answer   : {s['answer'][:200]}...")

---
## Step 5 — Format Train / Validation Split

**`formatter.py`** converts raw QA pairs into Gemma 4's chat template format and splits
into train (90%) and validation (10%) sets.

Each training example wraps the QA pair in the full token sequence:
```
<bos><|system|>...policy assistant prompt...<|end|>
<|user|>...question...<|end|>
<|assistant|><|channel>thought
...reasoning trace...
<channel|>
...answer with citation...<|end|>
```

Expected output: **~276 train + ~30 valid examples**

In [ ]:
%run scripts/formatter.py

In [ ]:
train = [json.loads(l) for l in open('data/train.jsonl')]
valid = [json.loads(l) for l in open('data/valid.jsonl')]

avg_tokens = sum(len(r['text'].split()) for r in train) // len(train)
print(f"Train examples : {len(train)}")
print(f"Valid examples : {len(valid)}")
print(f"Avg tokens     : ~{avg_tokens} words per example")

print("\n── Sample training example (truncated) ──")
print(train[0]['text'][:500] + "...")

---
## Step 6 — Fine-Tune with MLX LoRA

**`trainer.py`** runs LoRA fine-tuning via `mlx_lm` on the merged train set.

| Parameter | Value |
|-----------|-------|
| Base model | `mlx-community/gemma-4-E4B-it-4bit` |
| Method | LoRA (rank 8, α 16) |
| Iterations | 1500 |
| Learning rate | 1e-4 |
| Batch size | 4 |
| Hardware | Apple Silicon GPU (Metal / MLX) |

⚠️ **This step takes ~60–90 minutes on M-series chips.**
Checkpoints are saved every 100 iterations to `outputs/cbu-gemma4-e4b-mlx-v1/`.
If interrupted, re-run — the adapter will be resumed from the latest checkpoint.

> **Tip:** Watch for the lowest validation loss checkpoint.
> In our run, iter 800 (val loss 0.954) outperformed the final iter 1500 (val loss 1.512).

In [ ]:
%run scripts/trainer.py --iters 1500

In [ ]:
from pathlib import Path

adapter_dir = Path('outputs/cbu-gemma4-e4b-mlx-v1')
checkpoints = sorted(adapter_dir.glob('*_adapters.safetensors'))

print(f"Adapter directory : {adapter_dir}")
print(f"Checkpoints saved : {len(checkpoints)}")
for ck in checkpoints[-5:]:
    size_mb = ck.stat().st_size / 1e6
    print(f"  {ck.name}  ({size_mb:.1f} MB)")

final = adapter_dir / 'adapters.safetensors'
if final.exists():
    print(f"\nFinal adapter : {final.name}  ({final.stat().st_size/1e6:.1f} MB)")

---
## Step 7 — Evaluate ROUGE-L

**`evaluator.py`** measures how much the fine-tuned model improved over the base model
by running both on the 30 validation questions and scoring answers with ROUGE-L.

ROUGE-L (Longest Common Subsequence) measures word-level overlap between
a generated answer and the reference answer — scores range from 0 to 1.

⚠️ **This step loads the model twice (base + fine-tuned) and takes ~10–20 minutes.**

In [ ]:
%run scripts/evaluator.py

In [ ]:
import numpy as np

results = [json.loads(l) for l in open('eval/results.jsonl')]
base_scores = [r['base_rougeL'] for r in results]
ft_scores   = [r['ft_rougeL']   for r in results]
improved    = sum(1 for b, f in zip(base_scores, ft_scores) if f > b)
pct         = (np.mean(ft_scores) - np.mean(base_scores)) / np.mean(base_scores) * 100

print("=" * 50)
print("  ROUGE-L Evaluation Summary")
print("=" * 50)
print(f"  Examples evaluated : {len(results)}")
print(f"  Base model avg     : {np.mean(base_scores):.4f}")
print(f"  Fine-tuned avg     : {np.mean(ft_scores):.4f}")
print(f"  Improvement        : {pct:+.1f}%")
print(f"  Examples improved  : {improved}/{len(results)}")
print("=" * 50)

---
## Step 8 — Plot Results by Question Type

**`plot_eval.py`** groups the 30 ROUGE-L scores by question type and generates
a grouped bar chart comparing base model vs fine-tuned model per category.

In [ ]:
%run scripts/plot_eval.py --out eval/rouge_chart.png

In [ ]:
from IPython.display import Image, display
display(Image('eval/rouge_chart.png'))

---
## Step 9 — Export GGUF for Ollama

**`burn_gguf.py`** merges the LoRA adapter back into the base model and exports
a quantized GGUF file for local deployment with Ollama.

Flow:
```
mlx_lm fuse  →  convert_hf_to_gguf.py  →  llama-quantize  →  Modelfile
(dequantize     (HF weights → BF16       (BF16 → Q4_K_M)
 + merge)        GGUF)
```

**Prerequisite:** Install Homebrew llama.cpp:
```bash
brew install llama.cpp
pip install gguf==0.19.0
```

⚠️ **The fuse step downloads the full base model weights (~9 GB) and takes ~15–20 minutes.**
The final GGUF will be ~5 GB (Q4_K_M quantization).

In [ ]:
%run scripts/burn_gguf.py

In [ ]:
burns = list(Path('burns').rglob('*.gguf'))
for f in burns:
    print(f"{f}  ({f.stat().st_size/1e9:.2f} GB)")

modelfile = Path('burns/cbu-gemma4-e4b-mlx-v1/Modelfile')
if modelfile.exists():
    print("\n── Modelfile ──")
    print(modelfile.read_text())

---
## Step 10 — Deploy with Ollama

Register and run the model locally:

In [ ]:
import subprocess
result = subprocess.run(
    ['ollama', 'create', 'cbu-faculty-gemma4', '-f', 'Modelfile'],
    cwd='burns/cbu-gemma4-e4b-mlx-v1',
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
print("\nRun your model with:")
print("  ollama run cbu-faculty-gemma4")

---
## Summary

| Stage | Output | Notes |
|-------|--------|-------|
| Chunk | 90 policy sections | 11 categories, avg 290 words |
| Enrich | Metadata tags | Audience, topic, entities |
| Generate | 300 QA pairs | 8 question types, reasoning traces |
| Format | 276 train / 30 valid | Gemma 4 chat template |
| Train | LoRA adapter | Best checkpoint: iter 800 (val loss 0.954) |
| Evaluate | ROUGE-L +77.4% | 28/30 examples improved |
| Export | 5.0 GB GGUF | Q4_K_M, ready for Ollama |

**What makes this pipeline different from basic fine-tuning:**
- LLM-generated QA pairs (not rule-based keyword matching)
- Reasoning traces in every training example (`<|channel>thought` format)
- Audience-aware system prompts per training example
- Policy citation format enforced in every answer

**Recommendations for improvement:**
- Use the best checkpoint (`0000800_adapters.safetensors`) instead of the final adapter before burning GGUF
- Add human evaluation on top of ROUGE-L — it does not catch hallucinated policy numbers
- Expand QA pairs targeting under-represented types (Comparative: n=1)